# SignAI - Text2Sign RAG
Convierte texto en español a secuencia de videos de señas LSM usando Groq.

In [1]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

VIDEOS_DIR = '/content/drive/MyDrive/signai/Videos_Listos'

video_files = list(Path(VIDEOS_DIR).glob('*.mp4')) + list(Path(VIDEOS_DIR).glob('*.mov'))

video_db = {}
for vf in video_files:
    key = vf.stem.lower().strip()
    video_db[key] = str(vf)

print(f'Videos disponibles: {len(video_db)}')
for k in sorted(video_db.keys()):
    print(f'  [{k}]')

Mounted at /content/drive
Videos disponibles: 106
  [a]
  [abuela]
  [abuelo]
  [adios]
  [ahora]
  [amar]
  [audifonos]
  [ayer]
  [b]
  [bicicleta]
  [bien]
  [bienvenido]
  [buenas noches]
  [buenas trades]
  [buenos dias]
  [c]
  [camara]
  [camion]
  [camisa]
  [carro]
  [como]
  [como estas]
  [como te llamas]
  [computadora]
  [correr]
  [cuaderno]
  [cual]
  [cuando]
  [cuanto vale]
  [cuantos]
  [d]
  [de nada]
  [disculpa]
  [donde]
  [duda]
  [e]
  [ejegir]
  [el ella]
  [encontrar]
  [enfermar]
  [ensenar]
  [entender]
  [escuela]
  [estar]
  [explicar]
  [f]
  [familia]
  [felicitar]
  [firmar]
  [fracasar]
  [g]
  [gorra]
  [gracias]
  [gracias (1)]
  [gusto conocerte]
  [h]
  [hermano]
  [hijo]
  [hola]
  [hola (1)]
  [hoy]
  [i]
  [j]
  [k]
  [l]
  [m]
  [mal]
  [mama]
  [moto]
  [n]
  [no]
  [nosotros]
  [nunca]
  [ñ]
  [o]
  [odio]
  [otro]
  [p]
  [pantalones]
  [papa]
  [perdon]
  [policia]
  [por favor]
  [por que]
  [q]
  [que milagro]
  [quien]
  [r]
  [s]
  [si

In [6]:
from groq import Groq
import json

GROQ_API_KEY = 'YOUR_GROQ_API_KEY_HERE'
client = Groq(api_key=GROQ_API_KEY)

available_signs = sorted(video_db.keys())
signs_list = ', '.join(available_signs)

SYSTEM_PROMPT = f"""Eres un experto en Lengua de Senas Mexicana (LSM).
Tu tarea es convertir texto en espanol a una secuencia de senas LSM.

SENAS DISPONIBLES EN LA BASE DE DATOS:
{signs_list}

REGLAS:
1. Convierte la frase al orden gramatical de LSM (Sujeto-Objeto-Verbo)
2. Usa SOLO senas de la lista disponible
3. Omite articulos (el, la, los, las, un, una)
4. Omite preposiciones que no tienen sena (de, en, con)
5. Usa forma base de verbos (estar, no estas)
6. Si una palabra no esta disponible, usa el sinonimo mas cercano de la lista
7. Responde UNICAMENTE con JSON con este formato:
{{"glosa": ["sena1", "sena2"], "traduccion_literal": "explicacion"}}

EJEMPLO:
Input: "Buenos dias, como estas tu?"
Output: {{"glosa": ["buenos dias", "como estas", "tu"], "traduccion_literal": "BUENOS-DIAS COMO-ESTAR TU"}}"""


def text_to_gloss(text):
    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': text}
        ],
        temperature=0.1,
        max_tokens=200
    )
    raw = response.choices[0].message.content.strip()
    raw = raw.replace('```json', '').replace('```', '').strip()
    try:
        return json.loads(raw)
    except:
        print(f'Error parsing: {raw}')
        return None


def find_video(sign):
    sign_norm = sign.lower().strip()
    if sign_norm in video_db:
        return video_db[sign_norm]
    for key, path in video_db.items():
        if sign_norm in key or key in sign_norm:
            return path
    return None


def translate_to_signs(text):
    print(f'Input: "{text}"')
    print('-' * 50)

    result = text_to_gloss(text)
    if not result:
        print('Error al procesar con LLM')
        return []

    glosa = result.get('glosa', [])
    print(f'Glosa LSM: {glosa}')
    print(f'Traduccion: {result.get("traduccion_literal", "")}')
    print()

    videos_encontrados = []
    for sena in glosa:
        video_path = find_video(sena)
        if video_path:
            print(f'  OK "{sena}" -> {Path(video_path).name}')
            videos_encontrados.append({'sena': sena, 'video': video_path, 'found': True})
        else:
            print(f'  NO ENCONTRADO: "{sena}"')
            videos_encontrados.append({'sena': sena, 'video': None, 'found': False})

    found = sum(1 for v in videos_encontrados if v['found'])
    print(f'Encontrados: {found}/{len(glosa)}')
    return videos_encontrados


print('Sistema RAG listo.')

Sistema RAG listo.


In [7]:
from IPython.display import display, HTML
import base64

def show_sign_videos(videos_list):
    found_videos = [v for v in videos_list if v['found']]
    if not found_videos:
        print('No hay videos para mostrar')
        return

    html_parts = ['<div style="display:flex;flex-wrap:wrap;gap:10px;">']
    for v in found_videos:
        with open(v['video'], 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        ext = Path(v['video']).suffix.lower()
        mime = 'video/mp4' if ext == '.mp4' else 'video/quicktime'
        html_parts.append(f'''
        <div style="text-align:center;border:1px solid #ddd;padding:8px;border-radius:8px;">
            <p style="font-weight:bold;margin:0 0 5px 0;">{v["sena"].upper()}</p>
            <video width="200" controls loop>
                <source src="data:{mime};base64,{b64}" type="{mime}">
            </video>
        </div>''')
    html_parts.append('</div>')
    display(HTML(''.join(html_parts)))

print('Reproductor listo.')

Reproductor listo.


In [8]:
# CAMBIA ESTA FRASE
frase = 'Hola, buenos dias. Como estas tu?'

videos = translate_to_signs(frase)
show_sign_videos(videos)

Output hidden; open in https://colab.research.google.com to view.

In [9]:
# PRUEBAS CON MULTIPLES FRASES
frases_test = [
    'Hola, como te llamas?',
    'Por favor, disculpa',
    'Gracias, de nada',
    'Cuanto vale?',
    'Yo no entiendo',
    'Quien eres tu?',
]

for frase in frases_test:
    videos = translate_to_signs(frase)
    encontrados = [v['sena'] for v in videos if v['found']]
    print(f'Secuencia: {encontrados}')
    print()

Input: "Hola, como te llamas?"
--------------------------------------------------
Glosa LSM: ['hola', 'como te llamas']
Traduccion: HOLA COMO-TE-LLAMAR

  OK "hola" -> Hola.mov
  OK "como te llamas" -> Como te llamas.mp4
Encontrados: 2/2
Secuencia: ['hola', 'como te llamas']

Input: "Por favor, disculpa"
--------------------------------------------------
Glosa LSM: ['perdon']
Traduccion: PERDON

  OK "perdon" -> Perdon.mp4
Encontrados: 1/1
Secuencia: ['perdon']

Input: "Gracias, de nada"
--------------------------------------------------
Glosa LSM: ['gracias', 'de nada']
Traduccion: GRACIAS DE-NADA

  OK "gracias" -> Gracias.mov
  OK "de nada" -> De nada.mov
Encontrados: 2/2
Secuencia: ['gracias', 'de nada']

Input: "Cuanto vale?"
--------------------------------------------------
Glosa LSM: ['cuanto vale']
Traduccion: CUANTO-VALE

  OK "cuanto vale" -> Cuanto vale.mov
Encontrados: 1/1
Secuencia: ['cuanto vale']

Input: "Yo no entiendo"
-------------------------------------------------